# Example 0.13 / 2: Check the Answer by Hand

Companion notebook for **0.13 Practice Exercises** in the AMPL lab Documentation, section "Check the Answer by Hand". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Check the Answer by Hand

The exercise asks for a hand check. A magazine page reaches 1.0 / 10 = 0.1 million people per
thousand dollars and a TV minute 1.8 / 20 = 0.09, so every dollar above the required TV goes to
magazines. The 10 required TV minutes cost 200, leaving 800 for 80 pages, and the audience is
1.8 × 10 + 80 = 98 million, the value the solver reports.

Part (b) adds a limit of 100 person-weeks of creative time, 3 per magazine page and 1 per TV
minute. It is one more parameter and constraint; the remaining parts change only the data.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set MEDIA;                          # advertising media
param audience {MEDIA} > 0;         # millions of people reached per unit
param cost {MEDIA} > 0;             # thousands of dollars per unit
param min_buy {MEDIA} >= 0;         # minimum units to buy
param budget > 0;                   # thousands of dollars
param weeks {MEDIA} >= 0;           # person-weeks of creative time per unit
param staff > 0;                    # person-weeks available

var Buy {m in MEDIA} >= min_buy[m]; # units bought

maximize Total_Audience: sum {m in MEDIA} audience[m] * Buy[m];  # millions reached

subject to Budget: sum {m in MEDIA} cost[m] * Buy[m] <= budget;  # spending within budget
subject to Creative: sum {m in MEDIA} weeks[m] * Buy[m] <= staff; # creative time within staff

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["MEDIA"] = ["TV", "Magazine"]
ampl.param["audience"] = {"TV": 1.8, "Magazine": 1.0}
ampl.param["cost"] = {"TV": 20, "Magazine": 10}
ampl.param["min_buy"] = {"TV": 10, "Magazine": 0}
ampl.param["budget"] = 1000
ampl.param["weeks"] = {"TV": 1, "Magazine": 3}
ampl.param["staff"] = 100

In [5]:
# run.py
ampl.solve(solver="highs")
print(ampl.solve_result, ampl.var["Buy"].to_dict(), ampl.obj["Total_Audience"].value())

HiGHS 1.14.0: optimal solution; objective 92
2 simplex iterations
0 barrier iterations
solved {'Magazine': 20, 'TV': 40} 92.0
